In [ ]:
###DIDNT WRITE THE FULL CODE IN THIS FILE 
from langchain_ollama import ChatOllama
from dotenv import load_dotenv 
load_dotenv() 
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

In [ ]:
import os 
generator_llm = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url = os.getenv("OLLAMA_BASE_URL"),
    headers = {"Authorization":f"Bearer {os.getenv("OLLAMA_API_KEY")}"}
)
evaluator_llm = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url = os.getenv("OLLAMA_BASE_URL"),
    headers = {"Authorization":f"Bearer {os.getenv("OLLAMA_API_KEY")}"}
)
optimiser_llm = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url = os.getenv("OLLAMA_BASE_URL"),
    headers = {"Authorization":f"Bearer {os.getenv("OLLAMA_API_KEY")}"}
) #make sure all the 3 things have the best llm model 


In [3]:
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage

In [5]:
class TweetState(TypedDict):
    topic:str
    tweet:str 
    evaluation: Literal['Accepted','Needs Improvement'] 
    feedback:str 
    iterations:int 
    max_iterations: int 

In [8]:
def generate_tweet(state:TweetState)->TweetState:
    messages = [
        SystemMessage(content="you are funny and cleaver Twitter/X influencer"),
        HumanMessage(content=f""" 
        Generate the tweet based on the topic {state['topic']}
        Rules:
        Do not use question answer format 
        use max 180 characters long 
        use simple day to day english 
        do not use any emojis """)
    ]
    response = generator_llm.invoke(messages).content 
    return {'tweet':response} 


In [10]:
def evaluate_tweet(state:TweetState)->TweetState:
    messages = [
        SystemMessage(content="""You are the evaluator of the tweets, You evaluate tweets basedd on the humour, sarcasm, " \
                                Originality """),
        HumanMessage(content=f"""
                            Evaluate the following tweet: {state['tweet']}
                            Auto reject the tweet if its not original, in question answer format
                            ###respond only if it is in structured format 
                            -evaluation: "approved" or "needs_improvement"
                            -feedback: one 4 line paragraph saying the strengths and weakness
                            -based on score respond with either "approved" or ""needs_improvement """)
    ]
    response = evaluator_llm.invoke(messages).content 
    return {'evaluation':response} 

In [ ]:
def optimise_tweet(state:TweetState)->TweetState:
    

In [ ]:
graph=StateGraph(TweetState)
graph.add_node('generate',generate_tweet) 
graph.add_node('evaluate',evaluate_tweet) 
graph.add_node('optimise',optimise_tweet) 
graph.add_edge(START,'generate')